# The Self-Tooling Agent: Dynamic Tool Generation with Codestral

**An agent that writes, tests, and registers its own tools at runtime — then reuses them.**

Unlike code interpreters that run throwaway scripts, this agent builds a **persistent, growing toolkit**.
Each tool is syntax-checked, tested, and iteratively fixed by Codestral before registration.

| | Code Interpreter | Self-Tooling Agent |
|---|---|---|
| Execution | Throwaway scripts | Persistent, reusable tools |
| Memory | Isolated per run | Tools accumulate across turns |
| Composition | None | Tool B can call Tool A |
| Validation | None | Syntax check → test → iterative fix |
| Output | Results only | Exportable Python module |

In [ ]:
!pip install mistralai -q

In [ ]:
import ast
import json
import os
import subprocess
import sys
import traceback
import textwrap
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Callable, Optional
from mistralai import Mistral

client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])

CODESTRAL = "codestral-latest"
ORCHESTRATOR = "mistral-large-latest"
MAX_FIX_ATTEMPTS = 3

## 1. Tool Registry

The core differentiator: tools persist, accumulate, and compose.

In [ ]:
@dataclass
class Tool:
    name: str
    description: str
    parameters: dict
    source_code: str
    function: Callable
    version: int = 1
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    log: list[str] = field(default_factory=list)


class ToolRegistry:
    def __init__(self):
        self.tools: dict[str, Tool] = {}
        self.history: list[dict] = []

    def register(self, tool: Tool):
        if tool.name in self.tools:
            tool.version = self.tools[tool.name].version + 1
        self.tools[tool.name] = tool
        self.history.append({"action": "registered", "tool": tool.name, "v": tool.version})

    def get(self, name: str) -> Optional[Tool]:
        return self.tools.get(name)

    def describe_all(self) -> str:
        if not self.tools:
            return "No tools available yet."
        lines = []
        for t in self.tools.values():
            params = ", ".join(f"{k}: {v.get('type','any')}" for k, v in t.parameters.items())
            lines.append(f"- {t.name}({params}) — {t.description} [v{t.version}]")
        return "\n".join(lines)

    def get_source(self, name: str) -> Optional[str]:
        tool = self.tools.get(name)
        return tool.source_code if tool else None

    def export_module(self) -> str:
        parts = ['\'\'\'Auto-generated toolkit by Self-Tooling Agent.\'\'\'\n']
        for t in self.tools.values():
            parts.append(f"# {t.name} v{t.version} — {t.description}")
            parts.append(t.source_code + "\n")
        return "\n".join(parts)


registry = ToolRegistry()

## 2. Tool Forge

Codestral generates tools through an iterative loop: **generate → syntax check → test → fix → repeat**.

In [ ]:
def clean_code(code: str) -> str:
    """Strip markdown fences from LLM output."""
    code = code.strip()
    if code.startswith("```"):
        code = code.split("\n", 1)[1] if "\n" in code else code[3:]
    if code.endswith("```"):
        code = code.rsplit("```", 1)[0]
    return code.strip()


def compile_function(source: str, name: str) -> Callable:
    """Compile source code into a callable."""
    ns = {}
    exec(source, ns)
    if name in ns:
        return ns[name]
    for v in ns.values():
        if callable(v) and not isinstance(v, type):
            return v
    raise RuntimeError(f"Function \'{name}\' not found in generated code")


def design_tool(need: str, existing_tools: str) -> dict:
    """Ask Codestral to design a tool spec."""
    resp = client.chat.complete(
        model=CODESTRAL,
        messages=[{"role": "user", "content": f"""Design a Python function for this need:
\"{need}\"

Existing tools (you can compose with these):
{existing_tools}

Respond in JSON:
{{"name": "func_name", "description": "what it does", "parameters": {{"param": {{"type": "str", "description": "..."}}}}, "uses_existing": []}}"""}],
        response_format={"type": "json_object"}
    )
    return json.loads(resp.choices[0].message.content)


def generate_code(spec: dict, existing_sources: str, error: str = None, prev_code: str = None) -> str:
    """Generate or fix function code with Codestral."""
    if error and prev_code:
        prompt = f"""Fix this function:
{prev_code}

Error: {error}

Return ONLY the corrected function code, no markdown."""
    else:
        compose_note = f"You can call these existing functions:\n{existing_sources}" if existing_sources else ""
        prompt = f"""Write a Python function:
- Name: {spec['name']}
- Description: {spec['description']}
- Parameters: {json.dumps(spec['parameters'])}
{compose_note}

Rules: include docstring, type hints, handle edge cases, put imports inside the function.
Return ONLY the function code, no markdown fences."""

    resp = client.chat.complete(model=CODESTRAL, messages=[{"role": "user", "content": prompt}])
    return clean_code(resp.choices[0].message.content)


def generate_tests(source: str) -> list[dict]:
    """Ask Codestral for test cases."""
    resp = client.chat.complete(
        model=CODESTRAL,
        messages=[{"role": "user", "content": f"""Given this function:
{source}

Generate 3 test cases as JSON: {{"tests": [{{"input": {{"param": "value"}}, "description": "what this tests"}}]}}
Include: normal case, edge case, error handling case."""}],
        response_format={"type": "json_object"}
    )
    result = json.loads(resp.choices[0].message.content)
    if isinstance(result, dict):
        for v in result.values():
            if isinstance(v, list):
                return v
    return result


# --- Package detection & user-approved installation ---

# Standard library modules (Python 3.11+) — not exhaustive but covers common ones
_STDLIB = {
    "abc", "aifc", "argparse", "array", "ast", "asyncio", "atexit", "base64",
    "binascii", "bisect", "builtins", "calendar", "cmath", "codecs", "collections",
    "colorsys", "configparser", "contextlib", "copy", "csv", "ctypes", "dataclasses",
    "datetime", "decimal", "difflib", "dis", "email", "enum", "errno", "faulthandler",
    "fileinput", "fnmatch", "fractions", "ftplib", "functools", "gc", "getpass",
    "gettext", "glob", "gzip", "hashlib", "heapq", "hmac", "html", "http",
    "imaplib", "importlib", "inspect", "io", "ipaddress", "itertools", "json",
    "keyword", "linecache", "locale", "logging", "lzma", "mailbox", "math",
    "mimetypes", "multiprocessing", "numbers", "operator", "os", "pathlib",
    "pickle", "platform", "pprint", "profile", "queue", "random", "re",
    "readline", "secrets", "select", "shelve", "shlex", "shutil", "signal",
    "site", "smtplib", "socket", "sqlite3", "ssl", "stat", "statistics",
    "string", "struct", "subprocess", "sys", "sysconfig", "tempfile", "textwrap",
    "threading", "time", "timeit", "token", "tokenize", "tomllib", "traceback",
    "types", "typing", "unicodedata", "unittest", "urllib", "uuid", "venv",
    "warnings", "wave", "weakref", "webbrowser", "xml", "xmlrpc", "zipfile",
    "zipimport", "zlib",
}


def detect_third_party_imports(source: str) -> set[str]:
    """Parse source code and return set of third-party (non-stdlib) package names."""
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return set()
    packages = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                top = alias.name.split(".")[0]
                if top not in _STDLIB:
                    packages.add(top)
        elif isinstance(node, ast.ImportFrom) and node.module:
            top = node.module.split(".")[0]
            if top not in _STDLIB:
                packages.add(top)
    return packages


def check_and_install_packages(packages: set[str], log: list[str]) -> bool:
    """Check which packages are missing, ask user, install if approved. Returns True if all ok."""
    missing = []
    for pkg in packages:
        try:
            __import__(pkg)
        except ImportError:
            missing.append(pkg)

    if not missing:
        return True

    log.append(f"   📦 Third-party packages needed: {', '.join(missing)}")
    answer = input(f"   ⚠️  Install packages {missing}? [y/n]: ").strip().lower()
    if answer in ("y", "yes"):
        for pkg in missing:
            log.append(f"   📥 Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            log.append(f"   ✅ {pkg} installed")
        return True
    else:
        log.append("   ❌ User declined package installation")
        return False

In [ ]:
def forge_tool(need: str) -> Tool:
    """Full pipeline: design → generate → validate syntax → test → fix → register."""
    log = []
    existing_desc = registry.describe_all()

    # Step 1: Design
    log.append("📐 Designing tool...")
    spec = design_tool(need, existing_desc)
    log.append(f"   → {spec['name']}: {spec['description']}")

    # Gather composable sources
    existing_sources = ""
    for t_name in spec.get("uses_existing", []):
        src = registry.get_source(t_name)
        if src:
            existing_sources += f"\n{src}\n"

    # Step 2: Generate + iterative syntax validation
    log.append("🔨 Generating code...")
    source = None
    last_error = None
    for attempt in range(MAX_FIX_ATTEMPTS + 1):
        source = generate_code(spec, existing_sources, error=last_error, prev_code=source)
        try:
            ast.parse(source)
            log.append(f"   ✅ Syntax valid (attempt {attempt + 1})")
            break
        except SyntaxError as e:
            last_error = f"Line {e.lineno}: {e.msg}"
            log.append(f"   ⚠️ Syntax error: {last_error}")
    else:
        raise RuntimeError("Failed to generate syntactically valid code")

    # Step 3: Detect third-party packages and ask user
    third_party = detect_third_party_imports(source)
    if third_party:
        if not check_and_install_packages(third_party, log):
            # User declined — regenerate using only standard library
            log.append("   🔄 Regenerating with stdlib only...")
            source = generate_code(
                spec, existing_sources,
                error=f"Do NOT use these packages: {', '.join(third_party)}. Use only the Python standard library.",
                prev_code=source
            )
            try:
                ast.parse(source)
                log.append("   ✅ Stdlib-only version generated")
            except SyntaxError as e:
                raise RuntimeError(f"Stdlib regeneration produced invalid syntax: {e}")
            # Re-check — if it still needs third-party, warn
            remaining = detect_third_party_imports(source)
            if remaining:
                log.append(f"   ⚠️ Still uses: {remaining} — tests may fail")

    # Step 4: Generate test cases
    log.append("🧪 Generating tests...")
    tests = generate_tests(source)
    log.append(f"   → {len(tests)} test cases")

    # Step 5: Test + iterative fix loop
    log.append("🏃 Running tests...")
    for attempt in range(MAX_FIX_ATTEMPTS + 1):
        func = compile_function(source, spec["name"])
        failures = []
        for i, test in enumerate(tests):
            try:
                func(**test["input"])
                log.append(f"   ✅ Test {i+1}: {test['description']}")
            except Exception as e:
                failures.append(f"Input: {test['input']} → {type(e).__name__}: {e}")
                log.append(f"   ❌ Test {i+1}: {test['description']} — {e}")

        if not failures:
            break
        if attempt < MAX_FIX_ATTEMPTS:
            log.append(f"   🔧 Fixing {len(failures)} failure(s)...")
            source = generate_code(spec, existing_sources, error="\n".join(failures), prev_code=source)
            try:
                ast.parse(source)
            except SyntaxError:
                continue

    func = compile_function(source, spec["name"])
    tool_name = spec["name"]
    log.append(f"🎉 Tool ready: {tool_name}")

    tool = Tool(
        name=spec["name"], description=spec["description"],
        parameters=spec.get("parameters", {}), source_code=source,
        function=func, log=log
    )
    registry.register(tool)
    return tool

## 3. The Agent

Mistral Large decides: use existing tool, create new one, or answer directly.

In [ ]:
def agent_decide(user_msg: str) -> dict:
    """Ask the orchestrator what action to take."""
    system = f"""You are a Self-Tooling Agent. You solve tasks by creating and using reusable Python tools.

Available tools:
{registry.describe_all()}

Respond in JSON with one of:
- {{"action": "USE_TOOL", "tool": "name", "args": {{}}, "reason": "..."}}
- {{"action": "NEED_TOOL", "description": "what the tool should do", "reason": "..."}}
- {{"action": "DIRECT", "answer": "..."}}"""

    resp = client.chat.complete(
        model=ORCHESTRATOR,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_msg}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(resp.choices[0].message.content)


def run(user_msg: str, depth: int = 0) -> str:
    """Process a user request. Creates tools on demand, reuses existing ones."""
    if depth > 2:
        return "Max depth reached."

    decision = agent_decide(user_msg)
    action = decision.get("action", "DIRECT")

    if action == "DIRECT":
        print("💬 Direct answer")
        return decision.get("answer", str(decision))

    elif action == "NEED_TOOL":
        print(f"🔧 Creating tool: {decision['description']}")
        tool = forge_tool(decision["description"])
        for line in tool.log:
            print(f"  {line}")
        print(f"\n📦 Registered: {tool.name} v{tool.version}")
        indent = "   "
        print(f"   Source:\n{textwrap.indent(tool.source_code, indent)}\n")
        return run(user_msg, depth + 1)

    elif action == "USE_TOOL":
        tool = registry.get(decision["tool"])
        if not tool:
            missing = decision["tool"]
            print(f"⚠️ Tool not found: {missing}, creating...")
            return run(user_msg, depth + 1)
        args = decision.get("args", {})
        print(f"⚡ Using: {tool.name}({args})")
        try:
            return str(tool.function(**args))
        except Exception as e:
            return f"Tool error: {e}"

    return str(decision)

## 4. Demo — Watch the Toolkit Grow

Starting from **zero tools**, the agent creates what it needs and reuses them.

In [ ]:
print(f"Tools at start: {len(registry.tools)}\n")

print("=" * 50)
print("📝 Request 1: Sentiment analysis")
print("=" * 50)
result = run("Analyze the sentiment of: 'The product is great but delivery was awful'")
print(f"Result: {result}\n")

In [ ]:
print("=" * 50)
print("📝 Request 2: Statistics")
print("=" * 50)
result = run("Calculate mean, median, and std dev of [23, 45, 12, 67, 34, 89, 56, 78, 21, 43]")
print(f"Result: {result}\n")

In [ ]:
print("=" * 50)
print("📝 Request 3: Reuse sentiment tool (no creation)")
print("=" * 50)
result = run("What's the sentiment of: 'I love this cookbook, it changed how I think about AI'")
print(f"Result: {result}\n")

In [ ]:
print("=" * 50)
print("📝 Request 4: Outlier detection")
print("=" * 50)
result = run("Find outliers in this data using IQR method: [2, 3, 5, 6, 7, 8, 9, 120, 3, 4]")
print(f"Result: {result}\n")

## 5. Inspect the Toolkit

In [ ]:
print(f"Total tools created: {len(registry.tools)}\n")
for name, tool in registry.tools.items():
    print("=" * 40)
    print(f"📦 {name} v{tool.version}")
    print(f"   {tool.description}")
    indent = "   "
    print(f"   Source:\n{textwrap.indent(tool.source_code, indent)}")
    print()

## 6. Export as Reusable Module

Unlike Code Interpreter, everything the agent built is exportable.

In [ ]:
module = registry.export_module()
print(module)

with open("generated_toolkit.py", "w") as f:
    f.write(module)
print("\n✅ Saved to generated_toolkit.py")